# 第13章 ミニGPTを鍛える：GPUで本格学習

漱石の小説7冊、100万字で、自分のGPTを育てる

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：夏目漱石の7作品（約100万字）で学習した、あなただけの「漱石GPT」。トップページで動いているのと同じ設定です
- **身につく数学・理論**：データを増やすと過学習が減る（データの大切さ） ／ GPUへの転送と混合精度（float16・autocast） ／ チェックポイント（学習途中の保存） ／ 学習曲線で2つの実験を比べる ／ マルチヘッドAttentionの役割分担
- **新しく使うPython・PyTorch**：`!wget` でファイルを取ってくる ／ 自作モジュールの `import` ／ `torch.autocast` と `GradScaler` ／ 辞書で設定をまとめて保存

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 2〜3時間（学習の待ち時間を含む） |
| 難易度 | ★★★★☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> いよいよ、第12章で組み立てたGPTを本格的に学習させます。データは『こころ』だけでなく、漱石の7作品・約100万字。無料のGPU（T4）なら数分で学習が終わり、あなたのGPTが「吾輩は」に続けて、漱石風の文章を書き始めます。記録表の検証損失も、ここで大きく更新されます。

> **🔗 これまでの復習とつながり：これまでの復習（第7・10・12章とのつながり）**
>
> この章のGPU学習ループは、これまで学んだ要素の総決算です。
>
> - **第7章（勾配降下の4ステップ）**：順伝播 `model(x, y)` → 勾配リセット `zero_grad` → 逆伝播 `backward()` → 更新 `opt.step()` の基本サイクル。
> - **第10章（AdamW・学習率スケジュール・Dropout・過学習の診察）**：ウォームアップ＋コサイン減衰、重み減衰 `weight_decay`、Dropout、そして訓練損失と検証損失のギャップから過学習を見抜く目。
> - **第12章（ミニGPTの設計図 `minigpt.py`）**：5つのクラス（`Head`〜`GPT`）を組み立て、1バッチ丸暗記テストまで済ませたあなたのGPTをそのまま読み込みます。

**ミナト**：ついに来ましたね、GPT！ 第1章のQwenと比べると、どれくらいの大きさなんですか？

**ボンゴ先生**：パラメータは約175万個。Qwen2.5-0.5Bの約280分の1だ。読ませる文章も100万字と、Qwenの数兆トークンに比べれば、太平洋に対するバスタブくらい。でも、仕組みはまったく同じGPTだ。バスタブでも、泳ぎの練習はできる。

## 準備：GPUのランタイムにする

この章は **GPU** を使います。Colabのメニューで「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから始めましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使う装置：", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
使う装置： cuda
Tesla T4
```

</details>

`cpu` と出た人も、この章のコードはそのまま動きます。ただし学習に1時間以上かかるので、GPUが使えるタイミング（混んでいない夜や早朝など）を狙うか、[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)のKaggleを使うのがおすすめです。

> **✨ コツ：環境・リソースとチェックポイントの目安**
>
> - **推奨環境**：Colab無料 T4 GPU または Kaggle GPU（実験1＋実験2合わせて **約5〜10分**）。CPUでも実行可能ですが完全版は **1時間以上** かかります（CPU短縮版 `max_iters=1000` なら約15分）。
> - **メモリ・ダウンロード量**：VRAM は混合精度で約 1〜2 GB、ダウンロードは青空文庫7作品＋`minigpt.py` の計約 2.5 MB。
> - **途中切断への備え**：無料Colabは長時間の放置や上限で切断されることがあるため、学習後はすぐ下の節でGoogleドライブへチェックポイント（`soseki_gpt.pt`）を保存します。

> **✨ コツ：この章の出力について**
>
> この章に載せている学習の出力は、教科書の検証用に**CPUで実際に実行した結果**です。GPUで実行すると、計算の順番や数の細かさ（後で説明する混合精度）の違いで、数値が少し変わります。小数点以下2桁目くらいまで似ていれば、正常です。

#### 📝 確認問題（`ch13-q01` · Lv.1 · device 自動判定とモデル・データ両方の .to(device)）

Colab の GPU 環境（`device = "cuda"`）で学習ループを回したところ、`RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!` が発生しました。原因と正しい直し方はどれですか？

- **a.** ミニバッチサイズ `batch_size` が GPU に乗らないほど大きすぎるため、`batch_size = 1` に減らして 1 サンプルずつ計算すればデバイスエラーが解消される
- **b.** 青空文庫から読み込んだテキストデータ `all_train` を `str` 文字列型のまま GPU へ転送し忘れているため、トークン ID 化の前に文字列ごと `.to(device)` を呼ぶ
- **c.** Colab に割り当てられた GPU が故障して正常に演算できない状態なので、`torch.cuda.is_available()` を `False` 扱いにして CPU だけで学習を回すように書き換える
- **d.** モデル（`GPT(...).to(device)`）とミニバッチのデータ（`x.to(device), y.to(device)`）の片方が CPU に残ったまま計算しようとしている。モデルと入力の両方を `.to(device)` で同じ装置へ移す

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：エラー文の中にある `at least two devices, cuda:0 and cpu` が何を意味しているか読み取りましょう。
- **ヒント 2**：PyTorch では同じ装置の上にあるテンソル同士しか計算できません。モデル本体と、入力データ `x, y`（評価時・生成時も含む）の両方に `.to(device)` が必要です。
- **正解**：`d`

**考え方**：PyTorch では異なるデバイス（CPU と GPU）にあるテンソル同士を直接演算できません。モデル生成時の `.to(device)` と、`get_batch` / `val_loss` / `write` 内の入力 `.to(device)` の両方を揃えるのが鉄則です。

1. `device = "cuda" if torch.cuda.is_available() else "cpu"` で使う装置を決める。
1. モデル：`model = GPT(...).to(device)`。
1. データ：`x.to(device), y.to(device)`（`get_batch` や `val_loss`、生成時の `idx` も忘れずに）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q01

</details>

## 第12章のGPTを取ってくる

第12章で一行ずつ書いたGPTのコードは、教科書のリポジトリに `minigpt.py` というファイルで置いてあります（中身は第12章のコードとまったく同じです）。`wget` というLinuxのコマンドでダウンロードして、`import` します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!wget -nv -O minigpt.py https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
import os
assert os.path.exists("minigpt.py") and os.path.getsize("minigpt.py") > 0, (
    "minigpt.py の取得に失敗しました。ネット接続を確認して再実行するか、付録Eを参照してください。"
)
from minigpt import GPT
print("ミニGPTの設計図を読み込みました")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ミニGPTの設計図を読み込みました
```

</details>

> **🐍 Python・PyTorchのポイント：自分のファイルを import する**
>
> `import torch` と同じように、同じフォルダにある `minigpt.py` は `import minigpt` や `from minigpt import GPT` で読み込めます。Pythonのファイル（**モジュール**）を分けておくと、ノートブックがすっきりします。Colabの左のフォルダアイコンを押すと、`minigpt.py` がダウンロードされているのが見えます。ダブルクリックすると中身も読めますよ。

## 漱石の7作品を読み込む

第9章までと同じ `load_soseki`（下のセルで定義し直します）を使って、7作品をまとめて読み込みます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, math, urllib.request

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

works = {
    "こころ": "773_ruby_5968",
    "坊っちゃん": "752_ruby_2438",
    "吾輩は猫である": "789_ruby_5639",
    "三四郎": "794_ruby_4237",
    "草枕": "776_ruby_6020",
    "夢十夜": "799_ruby_6024",
    "それから": "56143_ruby_50824",
}
texts = {}
for title, file_id in works.items():
    texts[title] = load_soseki(file_id)
    print(f"{title}：{len(texts[title]):,} 文字")

all_text = "\n\n".join(texts.values())
chars = sorted(set(all_text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print(f"合計：{len(all_text):,} 文字、語彙：{V} 文字")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
こころ：162,467 文字
坊っちゃん：88,997 文字
吾輩は猫である：321,104 文字
三四郎：171,888 文字
草枕：85,547 文字
夢十夜：16,197 文字
それから：174,641 文字
合計：1,020,853 文字、語彙：3665 文字
```

</details>

約102万字、語彙は3,665文字になりました。『こころ』だけのとき（2,061文字）より、使われる漢字が増えています。

次に、データを訓練用と検証用に分けます。これまでの章と成績を比べられるように、**検証データは『こころ』の最後の10%のまま**にしておきます。ほかの作品は、それぞれ前90%を訓練に使います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

kokoro = texts["こころ"]
nk = int(len(kokoro) * 0.9)
kokoro_train = encode(kokoro[:nk])
kokoro_val = encode(kokoro[nk:])          # 記録表と同じ検証データ

others = [encode(t[:int(len(t) * 0.9)]) for title, t in texts.items() if title != "こころ"]
all_train = torch.cat([kokoro_train] + others)
print("『こころ』だけの訓練データ：", len(kokoro_train))
print("7作品の訓練データ：        ", len(all_train))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
『こころ』だけの訓練データ： 146220
7作品の訓練データ：         918754
```

</details>

#### 📝 確認問題（`ch13-q02` · Lv.2 · 実験間の公平な比較（同一の検証データ kokoro_val を使う理由））

実験2では訓練データを漱石の7作品に増やしましたが、検証データにはあえて「**『こころ』の最後の10%（`kokoro_val`）**」をそのまま使い、さらに他の6作品からも前90%だけを取って `all_train` に混ぜています。検証データを『こころ』の最後の10%に固定する主な理由はどれですか？

- **a.** 『こころ』以外の 6 作品は青空文庫の文字コードや語彙構造が異なっており、同じ語彙表 `stoi` を使って検証損失を計算することが技術的にできないから
- **b.** 検証データ `kokoro_val` を訓練データ `all_train` の末尾にも混ぜて復習させることで、7 作品全体の文体を『こころ』寄りに微調整しやすくするため
- **c.** 実験 1（『こころ』だけ）やこれまでの章（第 5〜12 章）とまったく同じテスト問題（ものさし）で採点することで、データ増加の効果を公平に比較できるから
- **d.** 検証データの文字数が少ないほど 1 トークンあたりの交差エントロピー損失が構造上小さくなりやすく、実験 2 の成績を良く見せるのに有利だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：実験 1 と実験 2 で「訓練データ量」だけでなく「テスト問題（検証データ）」まで変えてしまうと、点数の変化を正しく解釈できるでしょうか？
- **ヒント 2**：変えたい条件（訓練データ）だけを変え、採点用のテスト問題（`kokoro_val`）を全実験で固定することで、純粋なデータ増加の効果を公平に比べられます。
- **正解**：`c`

**考え方**：科学実験（アブレーション研究）の大原則は「変えたい条件（訓練データ量）だけを変え、評価ベンチマーク（`kokoro_val`）を固定する」ことです。

1. 実験1：訓練＝『こころ』前90%（14.6万字）、検証＝`kokoro_val`。
1. 実験2：訓練＝7作品の前90%（91.9万字）、検証＝同じ `kokoro_val`。
1. 同じテスト問題で検証損失が `3.075 → 2.635` に下がったため、「データ増加が効いた」と明確に結論できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q02

</details>

## 学習の道具をそろえる

ミニバッチを作る関数、検証損失を測る関数、学習率スケジュール（第10章）を用意します。検証損失は、検証データを128文字ずつに区切って**全部**で測るので、毎回同じ値になります。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
block_size = 128    # 何文字前まで見るか（窓の大きさ）
batch_size = 32     # 1回に何個の窓を学習するか

def get_batch(data, g):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), generator=g)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def val_loss(model, data):
    model.eval()
    starts = range(0, len(data) - block_size - 1, block_size)
    xs = torch.stack([data[i:i + block_size] for i in starts])
    ys = torch.stack([data[i + 1:i + block_size + 1] for i in starts])
    total = 0.0
    for j in range(0, len(xs), 64):
        _, loss = model(xs[j:j + 64].to(device), ys[j:j + 64].to(device))
        total += loss.item() * len(xs[j:j + 64])
    model.train()
    return total / len(xs)

def lr_at(step, max_lr, total, warmup=200):
    if step < warmup:
        return max_lr * step / warmup
    progress = (step - warmup) / (total - warmup)
    return max_lr * 0.5 * (1 + math.cos(math.pi * progress))

`.to(device)` で、データをGPUのメモリに送っています。GPUで計算するには、**モデルとデータの両方**がGPUにいる必要があります。

<details><summary><b>基礎の補足：model.train() / model.eval() と torch.no_grad() の違い（なぜ両方必要？）</b>（クリックで開く）</summary>

`val_loss` 関数には **`@torch.no_grad()`**・**`model.eval()`**・**`model.train()`** の3つが登場します。初学者が最も混同しやすい2つのスイッチを整理しておきましょう。

| スイッチ | 何を切り替えるか | 忘れると何が起きるか |
|---|---|---|
| `model.eval()` / `model.train()` | **Dropout などの層の挙動**（評価中はランダムな遮断をオフにして全ユニットを使う） | `eval()` 忘れ → 検証損失がブレて悪化。末尾の `train()` 忘れ → 以後の訓練で Dropout が永遠にオフになり過学習！ |
| `@torch.no_grad()` / `with torch.no_grad():` | **微分用の計算グラフ記録**（逆伝播のための途中メモを保存しない） | 忘れ → 評価中も不要なメモがGPUメモリ（VRAM）を圧迫して遅くなる・メモリ不足の原因に |

この2つはまったく別のスイッチなので、検証や文章生成のときは**必ず両方セット**で使い、学習ループに戻る前に `model.train()` へ戻します。

また、`lr_at(step, max_lr, total, warmup=200)` は、最初の `warmup` ステップで `0 → max_lr` まで直線的に上げ（ウォームアップ）、その後は $\frac{1}{2}(1 + \cos(\pi \cdot \text{progress}))$ に従って滑らかに `max_lr → 0` へ下げる（コサイン減衰）関数です。たとえば `max_lr=0.001, total=1000, warmup=200` なら、`step=0` で `0.0`、`step=100` で `0.0005`、`step=200` で最大 `0.001`、`step=600`（減衰の中間点）で `0.0005`、`step=1000` で `0.0` になります。

</details>

#### 📝 確認問題（`ch13-q03` · Lv.2 · model.eval() / model.train() と torch.no_grad() の違いと両方必要な理由）

`val_loss` 関数では `@torch.no_grad()` を付けた上で、先頭で `model.eval()` を呼び、末尾で `model.train()` に戻しています。`model.eval()` と `@torch.no_grad()` の役割の違いとして正しいものはどれですか？

- **a.** `@torch.no_grad()` を付けると不要な計算が減ってパラメータ更新が高速化されるため、検証関数だけでなく訓練ループ本体にも必ずセットで付けるのが鉄則である
- **b.** `model.eval()` はモデル全体を自動的に GPU から CPU へ移動してメモリを空ける命令であり、`@torch.no_grad()` は評価前にモデルの重みを初期値へ戻す命令である
- **c.** `model.eval()` は Dropout などの層を評価モード（無効化）に切り替え、`@torch.no_grad()` は逆伝播用の計算グラフ記録を止めて高速・省メモリにする（役割が違うため両方必要）
- **d.** `model.eval()` と `@torch.no_grad()` はどちらも内部で勾配記録の停止だけを行うまったく同じ機能であり、可読性のために重ねているだけで片方消しても挙動は変わらない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「Dropout のオン・オフ」を切り替えるスイッチと、「微分用メモ（autograd）の記録」を止めるスイッチは同じものでしょうか、別々でしょうか？
- **ヒント 2**：`model.eval()` は Dropout などの層挙動を評価用へ切り替え、`@torch.no_grad()` は勾配用の計算グラフ記録を止めます。独立した役割なので両方セットで使います。
- **正解**：`c`

**考え方**：評価時に `model.eval()` を忘れると検証中にも Dropout がランダムに働いて損失がブレて悪化し、`torch.no_grad()` を忘れると不要な中間計算グラフで GPU メモリ（VRAM）を浪費します。

1. `@torch.no_grad()`：微分用の中間記録をオフ → 速く・VRAM節約（ただし Dropout はオンのまま）。
1. `model.eval()`：Dropout をオフ → 決定的な実力評価（ただし勾配記録はオンのまま）。
1. 末尾の `model.train()`：次の訓練ステップのために Dropout を再びオンに戻す！

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q03

</details>

#### 📝 確認問題（`ch13-q04` · Lv.3 · ウォームアップ付きコサイン学習率スケジュールの手計算）

本文の `lr_at(step, max_lr=0.004, total=2000, warmup=500)` について、ウォームアップ途中の **`step = 150`** のときの学習率はいくつになりますか？（小数で入力してください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`step < warmup` のウォームアップ区間では、学習率は `0` から `max_lr` までステップ数に比例して直線的に増えます。
- **ヒント 2**：式 `max_lr * step / warmup` に `max_lr = 0.004`、`step = 150`、`warmup = 500` を代入して計算しましょう（`150 / 500 = 0.3`）。
- **正解**：`0.0012`

**考え方**：ウォームアップ区間（`step < 500`）では学習率を `0` から `max_lr` まで直線的に上げていき、学習初期の不安定な勾配による発散を防ぎます。`0.004 * 150 / 500 = 0.0012` です。

1. `step = 150 < warmup = 500` なのでウォームアップ区間。
1. 進捗の割合：`150 / 500 = 0.3`。
1. `lr = 0.004 * 0.3 = 0.0012`（`1.2e-3`）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q04

</details>

*(この節の追加問題 `ch13-q05` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html で自動採点・復習できます)*

## 学習ループ（GPU対応版）

第10章の学習ループに、GPUで速く学習するための工夫（**混合精度**）を足したものです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def train(train_data, max_iters, lr=1e-3, seed=0, log_every=500):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = GPT(V, block_size=block_size, n_embd=128, n_head=4, n_layer=4, dropout=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    history = []
    for step in range(max_iters + 1):
        for group in opt.param_groups:
            group["lr"] = lr_at(step, lr, max_iters)
        x, y = get_batch(train_data, g)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)              # 勾配を0にする代わりに捨てる（第8章の W.grad = None と同じ）
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        if step % log_every == 0:
            vl = val_loss(model, kokoro_val)
            history.append((step, loss.item(), vl))
            print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証（こころ）{vl:.3f}")
    return model, history

#### 💡 理論のポイント：混合精度：数の細かさを半分にして速くする

コンピュータの小数（浮動小数点数）には、使うビット数によって種類があります。

| 形式 | ビット数 | 1個のバイト数 | 特徴 |
|---|---|---|---|
| float32（単精度） | 32 | 4 | PyTorchの標準。正確 |
| float16（半精度） | 16 | 2 | メモリ半分。T4などのGPUで特に速い。表せる範囲が狭い |
| bfloat16 | 16 | 2 | 表せる範囲はfloat32並み、細かさは粗い。新しいGPU向け |

`torch.autocast(…, dtype=torch.float16)` の中では、行列の掛け算などの重い計算を、自動的にfloat16で行います。パラメータそのもの（マスターウェイト）や、細かさが大事な計算（損失の合計など）はfloat32のまま。2つの精度を混ぜて使うので **混合精度**（mixed precision）と呼びます。T4ではfloat32の何倍も速く計算できます。

「なぜパラメータ自体も全部float16にしないの？」と思うかもしれません。理由は、パラメータの更新量（学習率 $\times$ 勾配）がごく小さな値（例えば `0.00001` など）になるからです。float16の小さな数にその微小な更新量を足そうとすると、桁落ちして更新が無視されてしまいます。そこで、パラメータの「原本（マスターコピー）」はfloat32で正確に保持し、計算速度が求められる重い行列の掛け算だけfloat16に任せる、という役割分担をしています。

float16は小さな数を表すのが苦手で、とても小さな勾配が0になってしまう（第5章のアンダーフロー！）ことがあります。そこで `GradScaler` が、損失を一時的に何千倍にもしてから `backward()` し、更新の前に元に戻します。勾配を「拡大鏡で見る」作戦です。

CPUのときは `enabled=False` なので、これらは何もしません。同じコードがCPUでもGPUでも動きます。

> **📘 基本情報とのつながり：浮動小数点数の形式**
>
> 基本情報で、浮動小数点数は「符号部・指数部・仮数部」でできていると習いましたね。float16は指数部5ビット・仮数部10ビット、bfloat16は指数部8ビット・仮数部7ビットです。指数部が多いほど**表せる範囲**が広く、仮数部が多いほど**細かさ**が高い。bfloat16は「範囲は広く、細かさは我慢」という、深層学習向けに考えられた形式です（Googleが考案しました）。第19章の量子化で、この話の続きをします。

<details><summary><b>基礎の補足：勾配クリッピング（clip_grad_norm_）はなぜ必要？ unscale_ との順番も確認</b>（クリックで開く）</summary>

学習ループの後半にある次の2行は、大規模LLMの学習で欠かさず入れられる「安全ベルト」です。

```python no-run
scaler.unscale_(opt)                                      # ① 先に拡大を元に戻す
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # ② 元の大きさの勾配を上限 1.0 で切る
```

**なぜ勾配爆発が起きるのか？**
深いネットワークの逆伝播では、たまたま難しいミニバッチに当たった瞬間などに勾配が極端に大きくなる（**勾配爆発**）ことがあります。そのまま更新すると1ステップで重みが吹き飛び、損失が急上昇したり `NaN` になったりします。

**L2 ノルムクリッピングの仕組み**：全パラメータの勾配を1本のベクトル $\boldsymbol{g}$ とみなし、長さ（L2 ノルム）$\|\boldsymbol{g}\|_2 = \sqrt{\sum_i g_i^2}$ を計算します。これが上限 `max_norm = 1.0` を超えたときだけ、全勾配に同じ係数 $\frac{1.0}{\|\boldsymbol{g}\|_2}$ を掛けて縮めます。

- たとえば $\boldsymbol{g} = [3.0, 4.0]$ なら $\|\boldsymbol{g}\|_2 = \sqrt{9 + 16} = 5.0 > 1.0$ なので、`1.0 / 5.0 = 0.2` 倍して `[0.6, 0.8]` にします（長さはちょうど `1.0`、**向きは `3:4` のまま不変**！）。
- $\|\boldsymbol{g}\|_2 \le 1.0$ なら何もしません。

なお、`scaler.scale(loss).backward()` の直後は勾配が数千倍に拡大されたままなので、**必ず先に `scaler.unscale_(opt)` で元の大きさに戻してから** `clip_grad_norm_` を呼ぶのが鉄則です。

</details>

#### 📝 確認問題（`ch13-q06` · Lv.2 · 混合精度（autocast + float32原本）と GradScaler の役割）

GPU 学習で float16 の混合精度を使うとき、`torch.amp.GradScaler` が必要な理由はどれですか？

- **a.** テキストデータの語彙数 `V` を float16 のビット幅に収まるよう 16 語彙単位に丸め込み、埋め込み層のメモリ転送量を削減するため
- **b.** CPU だけで計算するときに起きる丸め誤差を補正するための機能であり、GPU（`cuda`）環境では `enabled=False` として自動的に無効化される
- **c.** GPU の計算能力を引き出すために学習率を毎ステップ自動的に 2 倍ずつ拡大していき、通常より短いステップ数で最良の検証損失へ収束させるため
- **d.** float16 は非常に小さな数を表せず勾配が 0 に潰れる（アンダーフロー）ことがあるため、損失を一時的に大きく拡大してから逆伝播し、更新前に元の大きさに戻すため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：float16 で表現できる最小の正の数は約 $6 \times 10^{-5}$ です。それより小さな勾配が発生すると何が起きてしまうでしょうか？
- **ヒント 2**：小さな勾配が 0 に潰れる（アンダーフロー）のを防ぐため、`GradScaler` が損失を拡大鏡のように数千倍して逆伝播し、更新前に `unscale_` で割り戻します。
- **正解**：`d`

**考え方**：小さな勾配が float16 の表現限界を下回って 0 になる（アンダーフロー）のを防ぐため、`GradScaler` は「損失を数千倍して勾配を大きく保ったまま `backward()` し、更新前に `unscale_` で割り戻す」拡大鏡の役目を果たします。

1. `scaler.scale(loss).backward()`：損失を定数倍して勾配のアンダーフローを防ぐ。
1. `scaler.unscale_(opt)`：拡大した勾配を元のスケールへ割り戻す。
1. `scaler.step(opt)` / `scaler.update()`：もし `inf/nan` が出たら更新を飛ばして倍率を自動調整する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q06

</details>

#### 📝 確認問題（`ch13-q07` · Lv.2 · float16 と bfloat16 のビット配分（指数部 vs 仮数部）の違い）

同じ16ビット（2バイト）の浮動小数点数である **float16**（指数部5ビット・仮数部10ビット）と **bfloat16**（指数部8ビット・仮数部7ビット）の違いとして正しいものはどれですか？

- **a.** bfloat16 は float32 と同じ指数部 8 ビットを持つため「表せる数の範囲」が float32 並みに広くオーバーフロー／アンダーフローしにくい代わりに、細かさ（仮数部）は float16 より粗い
- **b.** bfloat16 は指数部へ 8 ビットを割り当てる代わりに符号部（1 ビット）を省略した形式であるため、正の数と 0 しか表現できず負の重みを扱えない
- **c.** float16 のほうが指数部のビット数が多くて表せる数の最大〜最小範囲が広く、bfloat16 のほうが仮数部のビット数が多くて有効桁数の細かさに優れている
- **d.** bfloat16 は精度を落とさずに広い範囲を表現するために 32 ビット（4 バイト）の記憶領域を使う形式であり、float16 に比べて 2 倍の GPU メモリを消費する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：浮動小数点数では「指数部」と「仮数部」がそれぞれ「数の範囲」と「有効数字の細かさ」のどちらを決めているか思い出しましょう。
- **ヒント 2**：どちらも合計 16 ビット（2 バイト）ですが、bfloat16 は指数部を float32 と同じ 8 ビットにしたため範囲が非常に広く、代わりに仮数部が 7 ビットと粗くなります。
- **正解**：`a`

**考え方**：深層学習の学習では「有効数字の細かい精度」よりも「勾配や損失がオーバーフロー／アンダーフローしない広い範囲」のほうが重要なため、新しいGPUでは bfloat16 が標準になっています。

1. float16：符号1 + 指数5 + 仮数10 = 16 bit（細かいが範囲が狭い → `GradScaler` が必要）。
1. bfloat16：符号1 + 指数8 + 仮数7 = 16 bit（範囲が float32 並みに広い → 学習が安定）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q07

</details>

*(この節の追加問題 `ch13-q08`, `ch13-q09`, `ch13-q10` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html で自動採点・復習できます)*

## 実験1：『こころ』だけで学習

まずは、これまでの章と同じ『こころ』だけで学習させます。これで、これまでのモデルとの公平な比較ができます。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
model_k, hist_k = train(kokoro_train, max_iters=3000)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ     0：訓練 8.217  検証（こころ）8.207
ステップ   500：訓練 3.089  検証（こころ）3.218
ステップ  1000：訓練 2.622  検証（こころ）3.081
ステップ  1500：訓練 2.303  検証（こころ）3.075
ステップ  2000：訓練 2.233  検証（こころ）3.075
ステップ  2500：訓練 2.099  検証（こころ）3.092
ステップ  3000：訓練 2.091  検証（こころ）3.091
```

</details>

## 実験2：7作品で学習

次に、同じモデル・同じ設定で、データだけを7作品（約6.3倍）に増やします。検証データは同じ『こころ』の最後の10%です。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
model_all, hist_all = train(all_train, max_iters=5000)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ     0：訓練 8.220  検証（こころ）8.207
ステップ   500：訓練 3.724  検証（こころ）3.465
ステップ  1000：訓練 3.371  検証（こころ）3.107
ステップ  1500：訓練 3.158  検証（こころ）2.962
ステップ  2000：訓練 3.157  検証（こころ）2.850
ステップ  2500：訓練 3.119  検証（こころ）2.767
ステップ  3000：訓練 2.972  検証（こころ）2.710
ステップ  3500：訓練 3.007  検証（こころ）2.679
ステップ  4000：訓練 2.832  検証（こころ）2.653
ステップ  4500：訓練 2.873  検証（こころ）2.638
ステップ  5000：訓練 2.876  検証（こころ）2.635
```

</details>

T4 GPUなら数分、CPUなら1時間ほどかかります。待っている間に、次の節を読んでおきましょう。

> **✨ コツ：GPUが使えない日は、短く試す**
>
> CPUしか使えないときは、`max_iters=5000` を `max_iters=1000` に変えれば、15分ほどで終わります。検証損失は上の表ほどは下がりませんが、「7作品のほうが、同じステップ数でも良くなる」傾向は確かめられます。完全版は、GPUが使える日にもう一度やればOKです。

## 大事なモデルを保存する

比べたり書かせたりする前に、**まず保存**しましょう。せっかく学習したモデルも、Colabのセッションが切れると消えてしまいます。保存先はGoogleドライブです（第10章と同じ手順）。今回は、モデルの中身だけでなく、**語彙と設定もいっしょに**保存します。第24章で、このファイルからブラウザで動くモデルを作ります。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import os
save_dir = "/content/drive/MyDrive/tsukuru-llm"
os.makedirs(save_dir, exist_ok=True)
checkpoint = {
    "model": model_all.state_dict(),
    "chars": chars,
    "config": {"vocab_size": V, "block_size": block_size, "n_embd": 128, "n_head": 4, "n_layer": 4, "dropout": 0.1},
}
torch.save(checkpoint, f"{save_dir}/soseki_gpt.pt")
print("保存しました：", round(os.path.getsize(f"{save_dir}/soseki_gpt.pt") / 1e6, 1), "MB")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
保存しました： 8.1 MB
```

</details>

> **✨ コツ：チェックポイント**
>
> 学習の**途中**でも、定期的に保存しておくと安心です。これを **チェックポイント** と言います。Colabの無料版は、長い学習の途中で切断されることがあります。たとえば `train` 関数の中で、検証損失が過去最良を更新したときだけ `torch.save` するようにしておけば、切断されても「一番良かったモデル」は手元に残ります。大きなモデルの学習では、チェックポイントは命綱です。

> **✨ コツ：Kaggleで学習した人は**
>
> Kaggleには `google.colab` がないので、`drive.mount` のセルは飛ばし、保存のセルの `save_dir` を `"/kaggle/working"` に変えて実行します。保存した `soseki_gpt.pt` は、画面右の「Output」からダウンロードして、Googleドライブに `tsukuru-llm` というフォルダを作ってアップロードしておきましょう（第24章で使います）。

### 続きを別の日にやるとき

この章は長いので、ここで一度休んでもOKです。次の日に続きをやるときは、次の順にセルを実行してから、下のセルで保存したモデルを読み込みます。

1. 「準備：GPUのランタイムにする」「第12章のGPTを取ってくる」のセル
2. 「漱石の7作品を読み込む」の2つのセル（データと語彙を作り直す。保存したときと同じ語彙になります）
3. 「学習の道具をそろえる」「学習ループ（GPU対応版）」のセル（関数を作るだけなので、すぐ終わります）
4. 「大事なモデルを保存する」の節は、`drive.mount` のセル**だけ**（保存のセルは実行しない）

実験1・実験2の学習のセルと、保存のセルは**飛ばします**。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
save_dir = "/content/drive/MyDrive/tsukuru-llm"
ckpt = torch.load(f"{save_dir}/soseki_gpt.pt", map_location=device, weights_only=True)
model_all = GPT(**ckpt["config"]).to(device)     # 保存した設定で、同じ形のGPTを作る
model_all.load_state_dict(ckpt["model"])         # 学習した中身を流し込む
print("読み込みました。語彙は同じ？", ckpt["chars"] == chars)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
読み込みました。語彙は同じ？ True
```

</details>

`GPT(**ckpt["config"])` の `**` は、第3章の `model(**inputs)` と同じ「辞書の中身を名前つきの引数として渡す」書き方です。`GPT(vocab_size=3665, block_size=128, …)` と書いたのと同じ意味になります。なお、次の節の「2つの実験を比べる」グラフは、学習中の記録（`hist_k` と `hist_all`）が必要なので、休んだ後は描けません。その節は `!pip install` のセルだけ実行して、グラフは教科書の図で確かめましょう。その後の「書かせてみる」「ヘッドたちの役割分担を見る」は、そのまま実行できます。

<details><summary><b>基礎の補足：state_dict() チェックポイントに config・chars と map_location が必要な理由</b>（クリックで開く）</summary>

`model.state_dict()` が返すのは、**「層の名前の文字列 → 重みテンソル」の対応辞書だけ**です。Python のクラス定義（`class GPT`）も、モデルの寸法（`n_embd` や `n_layer`）も、文字とIDの対応表も含まれていません。

そのため、安全に復元できるチェックポイントには必ず次の3点セットを保存します。

1. **`"model": model.state_dict()`**：学習したすべての重みとバッファ（`tril` 等）
2. **`"config": {...}`**：復元時にまったく同じ形の箱 `GPT(**ckpt["config"])` を作るための設計寸法
3. **`"chars": chars`**：文字とトークンIDの対応表（これが1文字でもずれると生成結果が文字化けする）

さらに読み込み時に `map_location=device` を指定しておけば、「T4 GPU で保存した重みを、別の日に CPU 環境で読み込む」ときもデバイス不一致エラーを防げます。また `weights_only=True` を付けておくと、重み・基本データ型以外の任意コードが実行されるのを防げて安全です（PyTorch 2.6以降の推奨・既定設定）。

</details>

#### 📝 確認問題（`ch13-q11` · Lv.2 · state_dict() チェックポイントに config・chars と map_location が必要な理由）

学習したモデルを `torch.save` で保存するとき、`model_all.state_dict()` だけでなく **`chars`（語彙）と `config`（設定辞書）** も同じ辞書にまとめて保存しています。その理由として正しいものはどれですか？

- **a.** `state_dict()` には Python のクラス定義コード（`class GPT`）も丸ごと保存されているため、`config` や `chars` は人間があとから中身を確認するメモ用の飾りにすぎないから
- **b.** `state_dict()` には学習済みの重みテンソルしか入っていないため、再読み込み時に同じ形の `GPT(**ckpt["config"])` を作り、同じ文字↔ID対応（`chars`）を再現するために両方必要だから
- **c.** PyTorch の `torch.save` 関数はチェックポイント用辞書にキーがちょうど 3 つ揃っていないと直列化エラーを起こす仕様になっており、ダミーを含めて 3 キーにしているため
- **d.** チェックポイントのファイル容量の大半を占めるのは文字リスト `chars` であり、これを省略すると Google ドライブ上のファイルが空（0 MB）として扱われてしまうから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：重みテンソルの数値だけを渡されたとき、「何層・何次元の GPT の箱へ流し込むか」と「トークン ID 42 がどの日本語文字か」は分かるでしょうか？
- **ヒント 2**：`config` は復元時に同じ形の `GPT(**ckpt["config"])` を作るため、`chars` は学習時と 1 文字もずれない `stoi` / `itos` を復元するために必須です。
- **正解**：`b`

**考え方**：チェックポイントは「重み（`state_dict`）＋設計図の寸法（`config`）＋語彙表（`chars`）」の3点セットで保存するのが、後日や別環境（第24章のブラウザ版など）で確実に復元するためのベストプラクティスです。

1. `ckpt["config"]` を使って `GPT(**ckpt["config"])` で同じ形のモデルを作る。
1. `model.load_state_dict(ckpt["model"])` で重みを流し込む。
1. `ckpt["chars"]` で当時と同じ `stoi` / `itos` を復元する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q11

</details>

## 2つの実験を比べる

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

plt.figure(figsize=(6.5, 3.8))
for hist, label in [(hist_k, "『こころ』だけ"), (hist_all, "7作品")]:
    steps = [h[0] for h in hist]
    plt.plot(steps, [h[2] for h in hist], marker="o", label=label + "（検証）")
plt.axhline(3.177, color="gray", linestyle="--", label="第10章のMLP")
plt.xlabel("ステップ"); plt.ylabel("検証損失（こころ）")
plt.ylim(2.4, 4.5)
plt.legend(); plt.title("データを増やすと、同じモデルがもっと賢くなる")
plt.show()

![2つの実験の検証損失の比較。7作品で学習したほうが、最後まで検証損失が下がり続ける](https://sakunokyumei.github.io/llm-research-textbook/img/figs/13-train-gpt/cell12_0.png)

> **💡 理論のポイント：結局、データがいちばん効く**
>
> 第10章では、過学習を抑えるためにドロップアウトや重み減衰を使いました。でも、最も強力な治療法は **データを増やすこと** です。
>
> - 『こころ』だけ：14.6万字を何十周もするので、途中から丸暗記が始まる
> - 7作品：92万字あるので、同じ文章を見る回数が少なく、丸暗記しにくい。しかも、漱石の文体という「共通の特徴」をいろいろな文章から学べる
>
> 『こころ』の検証データには『こころ』の続きが書いてあるのに、『こころ』以外の作品を読ませたほうが、それを上手に予測できるようになる。不思議ですが、人間も同じですね。同じ作家の本をたくさん読むと、その作家の次の一文が予想できるようになります。
>
> LLMの研究で「モデルの大きさ」と同じくらい「**データの量と質**」が重視されるのは、このためです（第15章のスケーリング則で、もう少し詳しく話します）。

<details><summary><b>基礎の補足：訓練損失と検証損失の「ギャップ」で読み解くデータスケーリング効果</b>（クリックで開く）</summary>

2つの実験のログを「訓練損失」と「検証損失（こころ）」の差（ギャップ）で並べてみると、データ量の威力が一目で分かります。

| 実験 | 訓練データ量 | ステップ | 訓練損失 | 検証損失（こころ） | 状態 |
|---|---|---|---|---|---|
| 実験1（『こころ』のみ） | 14.6万字 | 1500 | 2.303 | 3.075 | ギャップ +0.77（過学習の始まり） |
| 実験1（『こころ』のみ） | 14.6万字 | 3000 | 2.091 | **3.091**（悪化） | ギャップ **+1.00**（丸暗記が進行） |
| 実験2（漱石7作品） | 91.9万字（6.3倍） | 3000 | 2.972 | 2.710 | 過学習せず着実に汎化 |
| 実験2（漱石7作品） | 91.9万字（6.3倍） | 5000 | 2.876 | **2.635**（大幅更新） | 最後まで検証損失が下がり続けた |

注目すべきは、**訓練損失だけ見れば実験1（2.091）の方が実験2（2.876）より低い**ことです。しかしそれは『こころ』を丸暗記した見せかけの点数にすぎません。未知の検証データでは実験2（2.635）が圧勝しており、「訓練損失の低さ＝賢さではない」ことが実測データで分かります。

</details>

#### 📝 確認問題（`ch13-q12` · Lv.3 · 訓練損失と検証損失のギャップから読み解くデータスケーリング効果）

まったく同じミニGPT（175万パラメータ）で学習した2つの実験の結果を比べます。
- **実験1（『こころ』のみ・14.6万字）**：ステップ3000で訓練 `2.091`・検証 `3.091`（ステップ1500の `3.075` から悪化開始）
- **実験2（7作品・91.9万字）**：ステップ5000で訓練 `2.876`・検証 **`2.635`**（最後まで下がり続けた）

同じモデルなのに、実験2のほうが『こころ』の検証損失が大きく良くなった理由として正しいものはどれですか？

- **a.** 実験 2 では 7 作品を連結する処理の中で検証データ `kokoro_val` も訓練データ `all_train` へ含めて学習したため、検証損失が大幅に下がったから
- **b.** 訓練データが約 6.3 倍に増えたことで同じ文章の丸暗記（過学習）が起きにくくなり、他の 6 作品からも漱石の文体や日本語の共通パターンを広く学べたから
- **c.** 実験 2 のほうが訓練損失自体も `2.091` より大きく下がっており、訓練データへの適合度が実験 1 を圧倒したことで検証損失も連動して下がったから
- **d.** 実験 2 ではデータ増加に合わせてモデルの層数を 4 層から 12 層・埋め込み次元も 3 倍へ拡張し、モデル容量を大幅に強化して学習したから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：実験 1 と実験 2 はまったく同じモデル（175 万パラメータ）と検証データ（`kokoro_val`）です。訓練損失と検証損失の差がどう変わったか注目しましょう。
- **ヒント 2**：実験 1 は 14.6 万字の丸暗記で過学習（訓練 2.09 vs 検証 3.09）していましたが、91.9 万字へ 6.3 倍増やしたことで過学習が抑えられ検証損失が大きく伸びました。
- **正解**：`b`

**考え方**：Dropout や重み減衰も有効ですが、過学習に対する最も強力な治療法は「データを増やすこと」です。同じ作家の他作品を読むことで、汎用的な文体・語彙・文法の知識が身につき、『こころ』の未知の続きの予測精度も劇的に上がります。

1. 実験1：14.6万字を何十周もするため、途中から訓練損失だけ下がり検証損失が `3.075 → 3.091` と悪化（過学習）。
1. 実験2：91.9万字（6.3倍）に増やすと丸暗記しにくくなり、検証損失が `2.635` まで下がり続けた。
1. 訓練損失が実験1より高く見えても、検証損失（本番力）がはるかに良いのが重要。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q12

</details>

## 書かせてみる

7作品で学習したGPTに、いろいろな書き出しを与えてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def write(model, start, n=150, temperature=0.8, seed=0):
    torch.manual_seed(seed)
    model.eval()
    idx = encode(start)[None].to(device)
    out = model.generate(idx, max_new_tokens=n, temperature=temperature)
    return "".join(itos[i] for i in out[0].tolist())

for start in ["吾輩は", "先生は", "　山路を登りながら"]:
    print(write(model_all, start))
    print("-" * 40)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩はあらず御客さんが近頃のうちに、あって、車屋へ帰りになって橋石でも婆さん、あたりするつもりではないから、元の御飯が食うと大きな声だけで、あんな食うじゃないか」
「御茶を飲んでしたか」
「うん、どうかも知れない。え？」
「ええ、よし子さんに御用はありません。この原口さんのおさんが、どこかへ行ってやっぱい
----------------------------------------
先生はある。そうかして先生のうちに、あっちからそうなりみをしているで、先生は、あたりすこぶる熊本がたいって、性の目を凹ました」「私はあの時のあいです」と言った。奥さんはまた聞いたが、「帽子のうちでも亡くなったのかい」と、しばらくすると、「あなたはどうしてもお宅から原口さんの事がないんですか」と与次郎のほう
----------------------------------------
　山路を登りながら、大きな木の中に近い寂として、あった、室の一方へ灯をとり抜いで、婆さんと車を向けた。
「ああなたは朝もう一つすが、何だか行くかい」と云った。代助は単簡に答えた。
「今度は何がたくって、大きくまで中梃なものだい」と、しばらくすると、額を手にした。
「そう。この原口さんの事がないんだ」
「わたしにしてい
----------------------------------------
```

</details>

> **🐍 Python・PyTorchのポイント：encode(start)[None]**
>
> `encode(start)` は、文字のIDを並べた1次元のテンソル（形 `[T]`）です。でもGPTは「何個かの文章をまとめて」受け取る形 `[B, T]` を期待しています。`[None]` を付けると、先頭に長さ1の軸が増えて `[1, T]`（文章1個ぶん）になります。第12章で `torch.tensor([[…]])` と、かっこを二重にしていたのと同じ意味です。

**ボンゴ先生**：どうだい、キミのGPTの「作品」は？ 第4章の「私は腎臓の運ばなた。」から、ずいぶん遠くまで来ただろう？ 文法はだいぶ整い、漱石らしい言い回しも出てくる。ただし、話の筋はまだ迷子だ。175万個のつまみと100万字では、このあたりが限界。ChatGPTが賢いのは、この何万倍ものつまみとデータがあるからなんだよ。

#### 📝 確認問題（`ch13-q13` · Lv.2 · コード予測：バッチ軸追加 [None] と **ckpt["config"] の動き）

生成時に `encode(start)[None]` のように1次元リスト `[T]` をバッチ次元つきの2次元 `[1, T]` に包む感覚と、`**config` 展開を確認します。次のコードの出力を予測してください。

```python
ids = [12, 45, 7]
batch = [ids]
print(len(batch), len(batch[0]))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`batch = [ids]` は、長さ 3 のリスト `ids` をもう 1 枚のリストで包んで 2 次元リスト（バッチサイズ 1）にしています。
- **ヒント 2**：外側のリスト `batch` の要素数が `len(batch)`、その先頭要素 `batch[0]`（＝`ids`）の長さが `len(batch[0])` になります。
- **期待される出力**：`1 3`

**考え方**：GPT の `forward(idx)` は常に `[B, T]`（バッチ数 × トークン長）の2次元テンソルを期待するため、1文章だけ生成するときも `[None]`（または `[ids]`）で `B = 1` の軸を足して `[1, T]` にします。

1. `ids = [12, 45, 7]`（長さ `T = 3`）。
1. `batch = [ids]` は `[[12, 45, 7]]`（形 `[1, 3]`）。
1. `len(batch) = 1`、`len(batch[0]) = 3` → `1 3`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q13

</details>

## 記録表

| モデル | 章 | パラメータ | 学習データ | 検証データの損失 |
|---|---|---|---|---|
| でたらめ | 5 | — | — | 7.631 |
| bigram（数える） | 5 | — | こころ | 3.581 |
| MLP（全部の治療） | 10 | 66万 | こころ | 3.177 |
| Attention 1ヘッド | 11 | 28万 | こころ | 3.31 |
| **ミニGPT（4層・4ヘッド）** | **13** | **約175万** | **こころ** | **3.075**（1500〜2000ステップ） |
| **ミニGPT（4層・4ヘッド）** | **13** | **約175万** | **漱石7作品** | **2.635**（5000ステップ） |

※ 第13章のモデルは語彙が3,665文字に増えているので、厳密には少しだけ不利な条件での比較です。数値は教科書の検証環境（CPU）での実測値です。GPUで実行したあなたの数値も、ぜひ表に書き足してください。

第10章のMLP（3.177）から、『こころ』だけのGPTで **3.075**、7作品のGPTで **2.635**。パープレキシティにすると、24択から約14択へ。数えるbigram（36択）から比べると、迷いが半分以下になりました。

#### 📝 確認問題（`ch13-q14` · Lv.3 · 記録表の検証損失からパープレキシティ（何択の迷い）への換算）

7作品で学習したミニGPTの検証損失は **`2.635`** でした。これをパープレキシティ（$e^{\text{loss}}$＝「だいたい何択で迷っているか」）に換算すると約何択になりますか？（参考：$\ln 10 \approx 2.303$、$e^{0.332} \approx 1.394$、誤差 $\pm 0.3$ まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パープレキシティは $\mathrm{PPL} = e^{\text{loss}}$ です。指数法則 $e^{a + b} = e^a \cdot e^b$ を使って `2.635` を2つの数に分解してみましょう。
- **ヒント 2**：$2.635 = 2.303 + 0.332$ と分けられるので、$e^{2.635} = e^{2.303} \times e^{0.332} \approx 10 \times 1.394$ を計算します。
- **正解**：`13.94` 択

**考え方**：交差エントロピー損失 $L$ に対してパープレキシティ $e^L$ を計算すると、モデルの迷い具合を「均等な何択クイズ相当か」という直感的な数に直せます。$e^{2.635} = e^{2.303} \cdot e^{0.332} \approx 10 \times 1.394 = 13.94$（約14択）です。

1. 検証損失 $L = 2.635 = 2.303 + 0.332$。
1. $e^{2.303} \approx 10$、$e^{0.332} \approx 1.394$。
1. パープレキシティ $= e^{2.635} \approx 10 \times 1.394 = 13.94$（約14択）。
1. でたらめ（2061択）→ bigram（36択）→ 治療済みMLP（24択）→ 7作品ミニGPT（約14択）と着実に進化！

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q14

</details>

## ヘッドたちの役割分担を見る

最後に、学習したGPTの4つのヘッドが、それぞれどこに注目しているかを見てみましょう（第11章の1ヘッドとの違いに注目）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja                          # グラフで日本語を表示する（上の !pip install が必要）

sample = "先生は奥さんの顔を見て、彼女に話しかけた。"
idx = encode(sample)[None].to(device)
model_all.eval()
with torch.no_grad():
    model_all(idx)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
heads = model_all.blocks[-1].attn.heads          # 最後の層の4つのヘッド
for h, ax in enumerate(axes):
    ax.imshow(heads[h].wei[0].float().cpu(), cmap="Oranges")
    ax.set_xticks(range(len(sample))); ax.set_xticklabels(list(sample), fontsize=7)
    ax.set_yticks(range(len(sample))); ax.set_yticklabels(list(sample), fontsize=7)
    ax.set_title(f"ヘッド {h}")
plt.tight_layout()
plt.show()

![最後の層の4つのヘッドの注目の仕方。ヘッド0は文の頭の「先」を、ヘッド1と2は「先生は」の「は」をよく見て、ヘッド3は自分の近くの文字に広く注目している](https://sakunokyumei.github.io/llm-research-textbook/img/figs/13-train-gpt/cell14_0.png)

ヘッドによって、注目の仕方が違います。上の図では、ヘッド0は文の頭の「先」を、ヘッド1と2は「先生は」の「は」（主語の区切り）をよく見ています。ヘッド3は、自分の近くの文字に広く注目しています。誰も役割を割り振っていないのに、学習の中で**自然に分業**が生まれているのです。あなたのモデルの図は、これとは違う分業になっているかもしれません。どのヘッドがどこを見ているか、観察してみましょう。

#### 🔬 やってみよう：GPUがあるなら、もっと大きく

GPUが使えるなら、モデルを大きくしてみましょう。`train` 関数はそのままで、次のセルを実行するだけです（大きなモデル用の `train_big` を新しく作ります）。このセルは `block_size` と `batch_size` を書き換えるので、必ず**上の保存が終わってから**試しましょう。後で保存のセルを実行し直すと、設定の記録がずれて、第24章で読み込めなくなります。

⚠️ **GPU専用セル（T4等）**：ランタイムがGPUであることを確認してから実行してください。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
block_size = 256
batch_size = 64

def train_big(train_data, max_iters, lr=6e-4, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = GPT(V, block_size=block_size, n_embd=384, n_head=6, n_layer=6, dropout=0.2).to(device)
    print("パラメータの数：", sum(p.numel() for p in model.parameters()))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))
    for step in range(max_iters + 1):
        for group in opt.param_groups:
            group["lr"] = lr_at(step, lr, max_iters)
        x, y = get_batch(train_data, g)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        if step % 500 == 0:
            print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証（こころ）{val_loss(model, kokoro_val):.3f}")
    return model

model_big = train_big(all_train, max_iters=5000)

パラメータは約1300万個。T4で20〜30分ほどかかります（`nvidia-smi` で、GPUが働いている様子も見てみましょう）。検証損失は、さらに下がるでしょうか？ それとも、データに対してモデルが大きすぎて、過学習するでしょうか？ 予想してから試してみてください。結果は、あなただけの研究データです。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`RuntimeError: Expected all tensors to be on the same device`**：モデルとデータの片方だけがGPUにいます。`model.to(device)` と、`get_batch` の中の `.to(device)` を確認しましょう。
> - **`CUDA out of memory`**：GPUのメモリが足りません。`batch_size` や `block_size` を半分にしてみましょう。それでもダメなら、「ランタイム」→「セッションを再起動する」でメモリを空にしてから、上から実行し直します。
> - **`No such file or directory: 'minigpt.py'`** / **`AssertionError: minigpt.py の取得に失敗しました`** / **`ModuleNotFoundError: No module named 'minigpt'`**：`wget` のセルを実行し忘れているか、ダウンロードに失敗しています。表示されたエラー内容（`ERROR 404` など）を確認して再実行してください。どうしてもダウンロードできないときは、第12章の5つのクラス（`Head` から `GPT` まで）のセルをこのノートブックに貼って実行し、`from minigpt import GPT` の行を消せば、同じように動きます。
> - **学習の途中で切断された**：無料のColabではよくあります。チェックポイントの節を参考に、途中経過を保存するようにしましょう。

#### 📝 確認問題（`ch13-q15` · Lv.4 · 【第3部総合】マルチヘッドの自然な役割分担とGPT自作の全体像）

【第3部の総まとめ】最後の層の4つの Attention ヘッドを可視化すると、人間が役割を指示していないのにヘッドごとに異なる注目パターン（文頭を見る係、主語の区切り「は」を見る係、近くを広く見る係など）が自然に生まれました。なぜ学習だけでこのような分業が起きるのか、また第11章→第12章→第13章で性能（検証損失 `3.31 → 3.097 → 2.635`）が段階的に向上した要因を3つの視点（構造・最適化/GPU・データ）から説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①ランダム初期値から始めた複数ヘッドが損失最小化でなぜ重複せず分業するのか、②第11→12→13章で効いた 3 つの軸（構造・学習/GPU・データ）を整理しましょう。
- **ヒント 2**：①異なる初期値のヘッドが `self.proj` で統合されるため別々の手がかりを拾うほうが損失が大きく下がる、②構造（1ヘッド→4層4ヘッドGPT）・学習/GPU（混合精度・コサインLR・クリッピング）・データ（1作品→7作品の6.3倍）の 3 要素でまとめます。
- **解答の目安**：マルチヘッドでは各ヘッドが異なるランダム初期値からスタートし、出力が連結されて `self.proj` で統合されるため、同じ場所ばかり見るヘッドが重複するよりも、別々の手がかり（文頭・助詞・直近の文脈など）を分担して拾ったほうが全体の次文字予測損失をより大きく下げられます。その結果、誤差逆伝播によって自然に役割分担が分化します。また第3部の性能向上は、(1) 構造面：1ヘッド Attention（第11章）から、マルチヘッド・FeedForward・残差接続・Pre-LN を4層積んだ GPT（第12章）への進化、(2) 学習・計算面：GPU＋混合精度（`autocast`/`GradScaler`）・コサイン学習率スケジュール・勾配クリッピングによる安定かつ長時間の学習、(3) データ面：1作品（14.6万字）から7作品（91.9万字）への6.3倍のデータ増加による過学習抑制と汎化向上、の3つの掛け算で達成されました。

**考え方**：モデル構造（Ch 11–12）・学習レシピと計算基盤（Ch 13前半）・データ量（Ch 13後半）の3本柱が揃って初めて強いLLMが育ちます。第4部ではここからトークナイザー（BPE）や本物のGPT-2へとさらにステップアップします。

1. 分業の仕組み：異なる初期値を持つ複数ヘッドが、`proj` 経由で損失を下げる方向に更新されるうち、互いに補完的な注目先へ自動分化する。
1. 構造の進化：1ヘッド（3.31）→ 4層4ヘッドGPT（こころ単体で3.075）。
1. 計算と安定化：GPU＋混合精度＋ウォームアップ・コサインLR＋勾配クリッピング。
1. データスケーリング：7作品（6.3倍）で過学習を抑え検証損失 `2.635`（約14択）へ到達。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html#ex-ch13-q15

</details>

**✅ できたらチェック**

- [ ] ColabのGPU（T4）でミニGPTを学習させた（またはCPUで完走した）
- [ ] 『こころ』だけの実験と、7作品の実験を比べた
- [ ] 「データを増やすと過学習が減る」を学習曲線で確かめた
- [ ] 自分のGPTに漱石風の文章を書かせた
- [ ] 語彙・設定ごとモデルをGoogleドライブに保存した
- [ ] 記録表の空欄を、自分の実験の数値で埋めた

> **📝 まとめ**
>
> - GPUで学習するには、**モデルとデータを `.to(device)`** で送る
> - **混合精度**（float16 ＋ `autocast` ＋ `GradScaler`）で、T4での学習が速くなる
> - 同じモデルでも、**データを増やす**と過学習が減り、検証損失が大きく下がる
> - 学習したモデルは、**語彙と設定もいっしょに**保存しておく（チェックポイント）
> - マルチヘッドAttentionの各ヘッドは、学習の中で自然に役割を分担する

**ボンゴ先生**：第3部、完走おめでとう！ キミはもう「GPTをゼロから作って学習させたことがある人」だ。世界中を探しても、そう多くはない。次の第4部では、自作GPTを「本物」に近づける。まずはトークナイザー。いまは1文字＝1トークンだけど、本物のLLMはBPEを使っていた（第2章）。それを自分で作るところからだ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/13-train-gpt.html
- **次の章**：[第14章 トークナイザーを自作する：BPE](https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html)